# 🔍 AraBERT — Arabic Hate Speech Detection
### Before vs After Fine-Tuning Comparison

| Setting | Value | Purpose |
|---|---|---|
| `MAX_LEN` | 64 | 4× faster attention vs 128 |
| `BATCH_SIZE` | 64/GPU | 2× GPU utilization |
| `GRAD_ACCUM` | 2 | Effective batch = 256 |
| `EPOCHS` | 3 + early stop | Cuts wasted epochs |
| `LR` | 3e-5 | Scaled for large effective batch |
| `LABEL_SMOOTH` | 0.1 | Regularization, better F1 |
| LLRD | decay=0.9 | Per-layer LR, better accuracy |

**Target: ~2h training on T4×2, higher accuracy than baseline AraBERT**

- No pip installs — Kaggle pre-installed packages only
- Works with any `transformers` version (4.x → 5.x)

---
## 1. Imports & Environment Check

In [ ]:
import os, re, unicodedata, inspect, warnings, logging
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
import transformers as _tf_mod

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()

print(f'transformers : {_tf_mod.__version__}')
print(f'torch        : {torch.__version__}')
print(f'device       : {device}')
print(f'GPUs         : {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}      : {torch.cuda.get_device_name(i)}')

# ── Detect renamed TrainingArguments params (transformers ≥ 4.41) ──
_ta_params = set(inspect.signature(TrainingArguments.__init__).parameters)
EVAL_STRATEGY_KEY = 'eval_strategy' if 'eval_strategy' in _ta_params else 'evaluation_strategy'
print(f'\nTrainingArguments eval key : {EVAL_STRATEGY_KEY}')

---
## 2. Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║                     USER CONFIGURATION                          ║
# ╠══════════════════════════════════════════════════════════════════╣

# ── Dataset path ───────────────────────────────────────────────────
# CSV must have TEXT_COLUMN and LABEL_COLUMN
# Labels: 0 = not hate speech, 1 = hate speech
DATASET_PATH = '/kaggle/input/datasets/houssemag1/arabic-arabert/balanced_arabic_dataset.csv'  # ← CHANGE THIS

# ── Column names ───────────────────────────────────────────────────
TEXT_COLUMN  = 'text'
LABEL_COLUMN = 'label'

# ── Model ──────────────────────────────────────────────────────────
# HF hub ID — Kaggle caches it automatically (no internet needed after first run)
MODEL_NAME = 'aubmindlab/bert-base-arabertv02'

# ╔══════════════════════════════════════════════════════════════════╗
# ║            OPTIMIZED HYPER-PARAMETERS (target: ~2h, T4×2)       ║
# ╠══════════════════════════════════════════════════════════════════╣
#
# Change from original → why it helps
# ─────────────────────────────────────────────────────────────────
# MAX_LEN        128 → 64   Arabic hate speech = short tweets/comments.
#                            Attention is O(n²): cutting seq_len in half
#                            gives ~4x faster attention, ~2x overall speedup.
#
# BATCH_SIZE      32 → 64   Doubles GPU utilization. T4 has 16 GB VRAM;
#                            BERT-base at seq=64 fits batch=64 comfortably in fp16.
#
# GRAD_ACCUM       1 → 2    Effective batch = 64×2×2GPUs = 256.
#                            Larger effective batch → smoother gradients
#                            → better generalization, no extra time cost.
#
# NUM_EPOCHS       4 → 3    3 epochs is sufficient for BERT fine-tuning.
#                            Early stopping (patience=1) cuts it further
#                            the moment val F1 stops improving.
#
# LEARNING_RATE 2e-5 → 3e-5 Linear scaling rule: LR should scale with
#                            effective batch size. 256/64 × 2e-5 = ~6e-5
#                            but 3e-5 is a safe sweet-spot for BERT.
#
# WARMUP_RATIO   0.1 → 0.06 Fewer total steps → shorter warmup needed.
#
# LABEL_SMOOTHING  0 → 0.1  Prevents over-confidence, acts as regularizer,
#                            measurably improves F1 on hate speech tasks.
#
# dataloader_num_workers 4 → 2  Kaggle limits CPU cores; 2 is safer.
# ─────────────────────────────────────────────────────────────────
# Combined effect: ~3-4× faster training + better accuracy than defaults
# ╚══════════════════════════════════════════════════════════════════╝

MAX_LEN       = 64      # was 128 — 4x faster attention
BATCH_SIZE    = 64      # was 32  — per GPU; 2x GPU utilization
GRAD_ACCUM    = 2       # effective batch = 64 × 2 × 2GPUs = 256
NUM_EPOCHS    = 3       # was 4   — early stopping handles the rest
LEARNING_RATE = 3e-5    # was 2e-5 — scaled for larger effective batch
WARMUP_RATIO  = 0.06    # was 0.1
WEIGHT_DECAY  = 0.01
LABEL_SMOOTH  = 0.1     # new — regularizes, improves F1
TEST_SIZE     = 0.15
VAL_SIZE      = 0.10

# ── Outputs ────────────────────────────────────────────────────────
OUTPUT_DIR = './arabert-finetuned'
LOGS_DIR   = './logs'

# ╚══════════════════════════════════════════════════════════════════╝

ID2LABEL = {0: 'Not Hate', 1: 'Hate Speech'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR,   exist_ok=True)

print('Configuration loaded ✅')
print(f'  MAX_LEN        : {MAX_LEN}')
print(f'  BATCH_SIZE     : {BATCH_SIZE} per GPU  →  effective {BATCH_SIZE * GRAD_ACCUM * 2} with accum+2GPUs')
print(f'  EPOCHS         : {NUM_EPOCHS}  (+ early stopping)')
print(f'  LEARNING_RATE  : {LEARNING_RATE}')
print(f'  LABEL_SMOOTHING: {LABEL_SMOOTH}')

---
## 3. Arabic Text Normalizer & Tokenizer

In [ ]:
def normalize_arabic(text):
    """Clean Arabic text: remove diacritics, normalize letters."""
    if not isinstance(text, str):
        return ''
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F]', '', text)   # tashkeel
    text = re.sub(r'[إأآا]', 'ا', text)                         # alef variants
    text = re.sub(r'[ىي]',   'ي', text)                         # ya variants
    text = re.sub(r'ة',      'ه', text)                         # ta marbuta
    text = re.sub(r'[^\u0600-\u06FF\u0750-\u077F\s\d]', ' ', text)  # non-Arabic
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer ready  |  vocab: {tokenizer.vocab_size:,}')

# Verify normalizer
_t = 'أَنْتُمْ قَوْمٌ لا تَسْتَحِقُّونَ'
print(f'\nNormalization test:')
print(f'  before: {_t}')
print(f'  after : {normalize_arabic(_t)}')

---
## 4. Evaluation Sample Texts (used Before AND After fine-tuning)

In [ ]:
SAMPLE_TEXTS = [
    # Hate Speech
    'أنتم قوم لا تستحقون العيش بيننا',
    'هؤلاء المهاجرون يدمرون بلدنا ويسرقون وظائفنا',
    'يجب طرد كل الأجانب من هذه الأرض',
    'لا مكان للكفار في مجتمعنا الإسلامي',
    'هذه الفئة من الناس لا تساوي شيئا',
    # Not Hate
    'أحب الطبيعة والجبال والبحر',
    'يجب أن نتعاون معا لبناء مستقبل أفضل',
    'التنوع الثقافي يثري المجتمعات البشرية',
    'قرأت كتابا رائعا عن تاريخ الحضارة العربية',
    'التعليم هو المفتاح الحقيقي للتقدم والازدهار',
    # Ambiguous
    'هذا الشعب له عادات غريبة جدا',
    'لا أفهم ثقافتهم ولا أريد أن أفهمها',
]
SAMPLE_LABELS = [1, 1, 1, 1, 1,  0, 0, 0, 0, 0,  1, 1]

print(f'✅ {len(SAMPLE_TEXTS)} evaluation samples ready')

---
## 5. Dataset Class & Shared Utilities

In [ ]:
class ArabicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = [normalize_arabic(t) for t in texts]
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(),
            'attention_mask' : enc['attention_mask'].squeeze(),
            'labels'         : torch.tensor(self.labels[idx], dtype=torch.long),
        }


def predict_batch(model, texts, batch_size=16):
    model.eval()
    all_logits = []
    preprocessed = [normalize_arabic(t) for t in texts]

    for i in range(0, len(preprocessed), batch_size):
        batch = preprocessed[i : i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt',
        ).to(device)
        with torch.no_grad():
            m = model.module if hasattr(model, 'module') else model
            out = m(**enc)
        all_logits.append(out.logits.cpu())

    logits    = torch.cat(all_logits, dim=0)
    all_probs = torch.softmax(logits, dim=-1).numpy()
    preds     = np.argmax(all_probs, axis=1).tolist()
    confs     = np.max(all_probs, axis=1).tolist()
    return preds, confs, all_probs


def show_results(texts, preds, confs, true_labels=None, title='Results'):
    rows = []
    for i, (text, pred, conf) in enumerate(zip(texts, preds, confs)):
        row = {
            '#'          : i + 1,
            'Text'       : (text[:52] + '…') if len(text) > 52 else text,
            'Prediction' : ID2LABEL[pred],
            'Confidence' : f'{conf:.1%}',
        }
        if true_labels is not None:
            row['Ground Truth'] = ID2LABEL[true_labels[i]]
            row['✓/✗']         = '✓' if pred == true_labels[i] else '✗'
        rows.append(row)
    df = pd.DataFrame(rows).set_index('#')
    print(f'\n{"─"*8} {title} {"─"*8}')
    print(df.to_string())
    if true_labels is not None:
        acc = accuracy_score(true_labels, preds)
        f1  = f1_score(true_labels, preds, average='macro', zero_division=0)
        print(f'\n  Accuracy : {acc:.1%}   |   Macro-F1 : {f1:.4f}')
    return df


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy'    : float(accuracy_score(labels, preds)),
        'f1_macro'    : float(f1_score(labels, preds, average='macro',    zero_division=0)),
        'f1_weighted' : float(f1_score(labels, preds, average='weighted', zero_division=0)),
    }

print('✅ Utilities ready')

---
## 6. PHASE 1 — AraBERT *Before* Fine-Tuning

> The LOAD REPORT warning you see here is **completely normal**:
> - `UNEXPECTED` keys = pre-training heads (MLM / NSP) — discarded, we don't need them
> - `MISSING` keys = `classifier.weight / bias` — freshly initialized, fine-tuning will train these

In [ ]:
print(f'Loading pre-trained AraBERT: {MODEL_NAME}')

model_before = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

if n_gpus > 1:
    model_before = nn.DataParallel(model_before)
    print(f'Wrapped with DataParallel ({n_gpus} GPUs)')

model_before = model_before.to(device)
print(f'✅ Model on device: {device}')

In [ ]:
preds_before, confs_before, probs_before = predict_batch(model_before, SAMPLE_TEXTS)

df_before = show_results(
    SAMPLE_TEXTS, preds_before, confs_before,
    true_labels=SAMPLE_LABELS,
    title='AraBERT BEFORE Fine-Tuning',
)

---
## 7. Load & Preprocess Dataset

In [ ]:
print(f'Loading: {DATASET_PATH}')
df_raw = pd.read_csv(DATASET_PATH)
print(f'Shape   : {df_raw.shape}')
print(f'Columns : {df_raw.columns.tolist()}')
print(df_raw[[TEXT_COLUMN, LABEL_COLUMN]].head(3))

df = df_raw[[TEXT_COLUMN, LABEL_COLUMN]].copy()
df.columns = ['text', 'label']
df = df.dropna().reset_index(drop=True)
df['label'] = df['label'].astype(int)
df['text']  = df['text'].apply(normalize_arabic)
df = df[df['text'].str.strip().ne('')].reset_index(drop=True)

print(f'\nClean shape : {df.shape}')
print(df['label'].value_counts().rename(index=ID2LABEL))

In [ ]:
df_tv, df_test = train_test_split(
    df, test_size=TEST_SIZE, random_state=SEED, stratify=df['label']
)
df_train, df_val = train_test_split(
    df_tv,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=SEED,
    stratify=df_tv['label'],
)

train_ds = ArabicDataset(df_train['text'].tolist(), df_train['label'].tolist(), tokenizer, MAX_LEN)
val_ds   = ArabicDataset(df_val['text'].tolist(),   df_val['label'].tolist(),   tokenizer, MAX_LEN)
test_ds  = ArabicDataset(df_test['text'].tolist(),  df_test['label'].tolist(),  tokenizer, MAX_LEN)

print(f'Train : {len(train_ds):,}  |  Val : {len(val_ds):,}  |  Test : {len(test_ds):,}')

---
## 8. Fine-Tuning

In [ ]:
print(f'Loading fresh model for fine-tuning: {MODEL_NAME}')

model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
    hidden_dropout_prob=0.1,        # standard dropout
    attention_probs_dropout_prob=0.1,
)

# ── Layer-wise Learning Rate Decay (LLRD) ──────────────────────────
# Lower layers (basic syntax) need smaller LR; upper layers (semantics) need more.
# LLRD consistently improves fine-tuning performance on classification tasks.
# Decay factor: each layer gets LR × DECAY^(num_layers - layer_idx)
def get_optimizer_grouped_params(model, base_lr=LEARNING_RATE, decay=0.9):
    no_decay   = ['bias', 'LayerNorm.weight']
    optimizer_grouped = []

    # Classification head — full LR
    optimizer_grouped.append({
        'params' : [p for n, p in model.named_parameters() if 'classifier' in n and p.requires_grad],
        'lr'     : base_lr,
        'weight_decay': 0.0,
    })

    # BERT layers — decayed LR
    num_layers = model.config.num_hidden_layers  # 12 for BERT-base
    for layer_idx in range(num_layers - 1, -1, -1):
        layer_lr = base_lr * (decay ** (num_layers - layer_idx))
        layer_prefix = f'bert.encoder.layer.{layer_idx}.'
        optimizer_grouped.append({
            'params': [p for n, p in model.named_parameters()
                       if layer_prefix in n and not any(nd in n for nd in no_decay) and p.requires_grad],
            'lr': layer_lr, 'weight_decay': WEIGHT_DECAY,
        })
        optimizer_grouped.append({
            'params': [p for n, p in model.named_parameters()
                       if layer_prefix in n and any(nd in n for nd in no_decay) and p.requires_grad],
            'lr': layer_lr, 'weight_decay': 0.0,
        })

    # Embeddings — smallest LR
    embed_lr = base_lr * (decay ** (num_layers + 1))
    optimizer_grouped.append({
        'params': [p for n, p in model.named_parameters()
                   if 'embeddings' in n and not any(nd in n for nd in no_decay) and p.requires_grad],
        'lr': embed_lr, 'weight_decay': WEIGHT_DECAY,
    })
    optimizer_grouped.append({
        'params': [p for n, p in model.named_parameters()
                   if 'embeddings' in n and any(nd in n for nd in no_decay) and p.requires_grad],
        'lr': embed_lr, 'weight_decay': 0.0,
    })
    return optimizer_grouped


# ── Custom Trainer with label smoothing loss ────────────────────────
# Label smoothing: instead of [0,1] targets we use [0.05, 0.95].
# This prevents over-confidence and improves F1 on imbalanced classes.
class SmoothedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


# ── Build TrainingArguments (version-safe) ──────────────────────────
training_args = TrainingArguments(
    output_dir                     = OUTPUT_DIR,
    logging_dir                    = LOGS_DIR,

    # ── Speed ────────────────────────────────────────────────────────
    num_train_epochs               = NUM_EPOCHS,
    per_device_train_batch_size    = BATCH_SIZE,       # 64 per GPU
    per_device_eval_batch_size     = BATCH_SIZE * 2,   # 128 per GPU for eval
    gradient_accumulation_steps    = GRAD_ACCUM,       # effective batch = 256
    fp16                           = True,             # AMP — ~2x speedup on T4
    dataloader_num_workers         = 2,                # safe on Kaggle CPU limits
    dataloader_pin_memory          = True,             # faster CPU→GPU transfer

    # ── Optimizer (LR handled by param groups above) ─────────────────
    learning_rate                  = LEARNING_RATE,
    warmup_ratio                   = WARMUP_RATIO,
    weight_decay                   = WEIGHT_DECAY,
    lr_scheduler_type              = 'cosine',
    max_grad_norm                  = 1.0,

    # ── Eval & Checkpointing ─────────────────────────────────────────
    save_strategy                  = 'epoch',
    load_best_model_at_end         = True,
    metric_for_best_model          = 'f1_macro',
    greater_is_better              = True,
    save_total_limit               = 2,               # keep only 2 checkpoints

    # ── Logging ──────────────────────────────────────────────────────
    logging_steps                  = 100,
    report_to                      = 'none',
    seed                           = SEED,

    **{EVAL_STRATEGY_KEY: 'epoch'},  # version-safe (detected in cell 1)
)

print(f'✅ TrainingArguments ready  (eval key = "{EVAL_STRATEGY_KEY}")')

# Build optimizer with LLRD
from torch.optim import AdamW
optimizer = AdamW(
    get_optimizer_grouped_params(model_ft),
    lr=LEARNING_RATE,
    eps=1e-8,
    betas=(0.9, 0.999),
)

trainer = SmoothedTrainer(
    model           = model_ft,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    tokenizer       = tokenizer,
    compute_metrics = compute_metrics,
    optimizers      = (optimizer, None),   # None = let Trainer build the scheduler
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=1)],
)

print('🚀 Fine-tuning started…')
print(f'   Effective batch size : {BATCH_SIZE * GRAD_ACCUM * n_gpus}')
print(f'   Max seq length       : {MAX_LEN}')
print(f'   Label smoothing      : {LABEL_SMOOTH}')
print(f'   LLRD decay factor    : 0.9  (per-layer LR decay)')
trainer.train()

---
## 9. Evaluate on Test Set

In [ ]:
print('Evaluating on hold-out test set…')
test_output = trainer.predict(test_ds)
test_preds  = np.argmax(test_output.predictions, axis=-1)

print('\n📊 Test Set Metrics:')
for k, v in test_output.metrics.items():
    print(f'  {k:<35} {v:.4f}')

print('\n📋 Classification Report:')
print(
    classification_report(
        df_test['label'].tolist(),
        test_preds,
        target_names=list(ID2LABEL.values()),
    )
)

---
## 10. PHASE 2 — AraBERT *After* Fine-Tuning (Same Samples)

In [ ]:
model_after = trainer.model
if n_gpus > 1:
    model_after = nn.DataParallel(model_after)
model_after = model_after.to(device)

preds_after, confs_after, probs_after = predict_batch(model_after, SAMPLE_TEXTS)

df_after = show_results(
    SAMPLE_TEXTS, preds_after, confs_after,
    true_labels=SAMPLE_LABELS,
    title='AraBERT AFTER Fine-Tuning',
)

---
## 11. Side-by-Side Comparison

In [ ]:
acc_before = accuracy_score(SAMPLE_LABELS, preds_before)
acc_after  = accuracy_score(SAMPLE_LABELS, preds_after)
f1_before  = f1_score(SAMPLE_LABELS, preds_before, average='macro', zero_division=0)
f1_after   = f1_score(SAMPLE_LABELS, preds_after,  average='macro', zero_division=0)

comparison = pd.DataFrame({
    'Text'        : [(t[:45] + '…') if len(t) > 45 else t for t in SAMPLE_TEXTS],
    'Ground Truth': [ID2LABEL[l] for l in SAMPLE_LABELS],
    'Before Pred' : [ID2LABEL[p] for p in preds_before],
    'Before Conf' : [f'{c:.1%}'  for c in confs_before],
    'Before ✓/✗'  : ['✓' if p == g else '✗' for p, g in zip(preds_before, SAMPLE_LABELS)],
    'After Pred'  : [ID2LABEL[p] for p in preds_after],
    'After Conf'  : [f'{c:.1%}'  for c in confs_after],
    'After ✓/✗'   : ['✓' if p == g else '✗' for p, g in zip(preds_after,  SAMPLE_LABELS)],
})

print('═' * 130)
print('  BEFORE vs AFTER Fine-Tuning')
print('═' * 130)
print(comparison.to_string(index=False))

print(f'\n{"─"*55}')
print(f'{"Metric":<20} {"Before FT":>12} {"After FT":>12} {"Δ":>9}')
print(f'{"─"*55}')
print(f'{"Accuracy":<20} {acc_before:>12.1%} {acc_after:>12.1%} {acc_after-acc_before:>+9.1%}')
print(f'{"Macro-F1":<20} {f1_before:>12.4f} {f1_after:>12.4f} {f1_after-f1_before:>+9.4f}')
print(f'{"─"*55}')

In [ ]:
sample_ids = [f'S{i+1}' for i in range(len(SAMPLE_TEXTS))]
x, w = np.arange(len(sample_ids)), 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle('AraBERT: Before vs After Fine-Tuning', fontsize=14, fontweight='bold')

b1 = ax1.bar(x - w/2, confs_before, w, label='Before FT', color='#E07B54', alpha=0.85)
b2 = ax1.bar(x + w/2, confs_after,  w, label='After FT',  color='#4C9BE8', alpha=0.85)
ax1.axhline(0.5, color='gray', linestyle='--', lw=1)
ax1.set_xticks(x); ax1.set_xticklabels(sample_ids, rotation=45)
ax1.set_ylim(0, 1.1); ax1.set_ylabel('Confidence')
ax1.set_title('Prediction Confidence per Sample'); ax1.legend()
for bar in list(b1) + list(b2):
    ax1.annotate(f'{bar.get_height():.0%}',
                 xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points',
                 ha='center', va='bottom', fontsize=7)

xi = np.array([0, 1])
ax2.bar(xi - w/2, [acc_before, acc_after], w, label='Accuracy', color='#E07B54', alpha=0.85)
ax2.bar(xi + w/2, [f1_before,  f1_after],  w, label='Macro-F1', color='#4C9BE8', alpha=0.85)
ax2.set_xticks(xi); ax2.set_xticklabels(['Before FT', 'After FT'])
ax2.set_ylim(0, 1.1); ax2.set_ylabel('Score')
ax2.set_title('Overall Accuracy & F1 Comparison'); ax2.legend()

plt.tight_layout()
plt.savefig('arabert_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved → arabert_comparison.png')

---
## 12. Save Fine-Tuned Model

In [ ]:
SAVE_PATH = './arabert-hate-speech-finetuned'
save_model = trainer.model
if hasattr(save_model, 'module'):
    save_model = save_model.module
save_model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'✅ Model saved → {SAVE_PATH}')